# Four Objects, Every Representation

A `semantic_object` class in this library is written once, in Python, as a small
dataclass with typed fields. Everything else - the SPARQL query that finds matching
instances, the SHACL shape that validates them, the BuildingMOTIF template that
builds them, and the RDF graph an instance actually produces - is *derived* from
that one definition.

The other two tutorials each look at one slice of this:
- `ontology-ingestion-tutorial.ipynb` - how the classes themselves get generated from
  the ontology's SHACL shapes.
- `s223-generated-classes-tutorial.ipynb` - the class API, walked one method at a
  time.

This notebook is a reference gallery instead: four concrete `s223` objects, chosen to
span the range of structural complexity in the model, each one shown across *all* of
its representations side by side, so the shape of the Python class and the shape of
everything downstream stay visibly connected.

| Object | What it adds |
|---|---|
| `Valve` | nothing - zero fields, the baseline case |
| `Duct` | one field, a fixed-vocabulary enumeration value (`medium`) |
| `Fan` | qualified fields - two differently-typed values through *one* relation |
| `Area` | a value property, not an entity - a float/unit pair instead of relations only |

In [1]:
import logging
logging.disable(logging.WARNING)  # buildingmotif is chatty at DEBUG/INFO

from semantic_objects.s223 import entities, properties, enumerationkinds
from semantic_objects.build_model import BMotifSession

print("Imports OK")

CRITICAL:root:Install the 'bacnet-ingress' module, e.g. 'pip install buildingmotif[bacnet-ingress]'


Imports OK


## A shared way to look at a class

Every class exposes the same three class-level methods regardless of how many fields
it has: `get_sparql_query()`, `generate_rdf_class_definition()` (SHACL), and
`__dataclass_fields__` for the raw field list. `describe_class()` below just prints
those together. The fourth representation - the RDF an *instance* produces - needs an
actual object, so `show_instance()` builds one through `BMotifSession`, the same
template-evaluation path used elsewhere in this library.

In [2]:
def describe_class(cls, include_hierarchy=False):
    print(f"### {cls.__name__}")
    if cls.comment:
        print(cls.comment.strip())
    print()
    print("Fields:", list(cls.__dataclass_fields__.keys()) or "(none)")
    print()
    print("--- SPARQL query (finds instances) ---")
    print(cls.get_sparql_query(ontology='s223'))
    print("--- SHACL shape (validates instances) ---")
    print(cls.generate_rdf_class_definition(include_hierarchy=include_hierarchy))


def show_instance(obj, ns='tutorial'):
    session = BMotifSession(ns=ns)
    session.evaluate(obj)
    print("--- RDF instance data ---")
    print(session.model.graph.serialize(format='turtle'))

## 1. `Valve` - the zero-field baseline

`Valve` declares no fields of its own at all - in the ontology, "being a valve" is
just a type assertion, with no relation that has to accompany it. This is the
floor every other example is measured against: a SPARQL query that's a single type
check, a SHACL shape with no `sh:property` at all, and an RDF instance that's a
single triple.

In [3]:
describe_class(entities.Valve)

### Valve
A piece of `Equipment` that can be adjusted to allow, regulate, or stop the flow of fluid in a pipe or a duct.

Fields: (none)

--- SPARQL query (finds instances) ---
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX s223: <http://data.ashrae.org/standard223#>
SELECT DISTINCT * WHERE { ?name rdf:type s223:Valve . }
--- SHACL shape (validates instances) ---
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix s223: <http://data.ashrae.org/standard223#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .

s223:Valve a s223:Class,
        rdfs:Class,
        sh:NodeShape ;
    rdfs:label "Valve" ;
    rdfs:comment "A piece of `Equipment` that can be adjusted to allow, regulate, or stop the flow of fluid in a pipe or a duct." ;
    rdfs:subClassOf s223:Equipment .




In [4]:
valve = entities.Valve()
valve._name = "Valve_101"
show_instance(valve)

{'name': rdflib.term.URIRef('urn:tutorial#Valve_101')}
--- RDF instance data ---
@prefix owl: <http://www.w3.org/2002/07/owl#> .

<urn:tutorial#> a owl:Ontology .

<urn:tutorial#Valve_101> a <http://data.ashrae.org/standard223#Valve> .




## 2. `Duct` - one field, a fixed-vocabulary value

`Duct.medium` must be a `Medium` - a value from the ontology's fixed `EnumerationKind`
vocabulary (`Air`, `Water`, `Refrigerant`, ...), not a free-form literal. That shows up
consistently: the SPARQL query adds a variable *and* a type check for it, the SHACL
shape gets an `sh:property` for `hasMedium`, and the RDF instance gets a second node
(`Fluid-Air`) alongside the `Duct` itself.

One wrinkle: `medium` is actually declared on `Duct`'s parent class, `Connection`, not
on `Duct` itself - `Duct` just inherits it. `generate_rdf_class_definition()` only
shows a class's *own* fields by default (so that subclassing to add one new field
doesn't reprint every inherited constraint too) - which means the call below shows
no property shape for `hasMedium` at all. Passing `include_hierarchy=True` shows the
full picture, inherited fields included.

In [5]:
print("Default (own fields only) - no hasMedium property shape shown:\n")
describe_class(entities.Duct)

Default (own fields only) - no hasMedium property shape shown:

### Duct
A `Connection` that is used to transport air such as supply, return, and exhaust in HVAC (Heating, Ventilation, and Air Conditioning) systems.

Fields: ['medium']

--- SPARQL query (finds instances) ---
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX s223: <http://data.ashrae.org/standard223#>
SELECT DISTINCT * WHERE { ?name rdf:type s223:Duct .
?medium rdf:type s223:Substance-Medium .
?name s223:hasMedium ?medium . }
--- SHACL shape (validates instances) ---
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix s223: <http://data.ashrae.org/standard223#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .

s223:Duct a s223:Class,
        rdfs:Class,
        sh:NodeShape ;
    rdfs:label "Duct" ;
    rdfs:comment "A `Connection` that is used to transport air such as supply, return, and exhaust in HVAC (Heating, Ventilation, and Air Conditioning) systems." ;
    rdfs:subClassOf s223:Connection 

In [6]:
print("include_hierarchy=True - hasMedium now appears, inherited from Connection:\n")
describe_class(entities.Duct, include_hierarchy=True)

include_hierarchy=True - hasMedium now appears, inherited from Connection:

### Duct
A `Connection` that is used to transport air such as supply, return, and exhaust in HVAC (Heating, Ventilation, and Air Conditioning) systems.

Fields: ['medium']

--- SPARQL query (finds instances) ---
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX s223: <http://data.ashrae.org/standard223#>
SELECT DISTINCT * WHERE { ?name rdf:type s223:Duct .
?medium rdf:type s223:Substance-Medium .
?name s223:hasMedium ?medium . }
--- SHACL shape (validates instances) ---
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix s223: <http://data.ashrae.org/standard223#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

s223:Duct a s223:Class,
        rdfs:Class,
        sh:NodeShape ;
    rdfs:label "Duct" ;
    rdfs:comment "A `Connection` that is used to transport air such as supply, return, and exhaust in HVAC (Heating, Ventilation, and Air

In [7]:
duct = entities.Duct(medium=enumerationkinds.Air())
duct._name = "Duct_101"
show_instance(duct)

{'name': rdflib.term.URIRef('urn:tutorial#Duct_101'), 'medium': rdflib.term.URIRef('urn:tutorial#Fluid-Air'), 'medium-_type': rdflib.term.Literal('Air')}
--- RDF instance data ---
@prefix ns1: <http://data.ashrae.org/standard223#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

<urn:tutorial#> a owl:Ontology .

<urn:tutorial#Duct_101> a ns1:Duct ;
    ns1:hasMedium <urn:tutorial#Fluid-Air> .

<urn:tutorial#Fluid-Air> a ns1:Substance-Medium .




## 3. `Fan` - qualified fields: two values through one relation

A `Fan` has an inlet and an outlet, but in the ontology both are reached through the
*same* relation, `hasConnectionPoint` - it's SHACL's qualified value shapes that
narrow one occurrence to an `InletConnectionPoint` and the other to an
`OutletConnectionPoint`. The ingestion pipeline turns that into two distinct typed
Python fields, `inlet_connection_point` and `outlet_connection_point`.

That's visible in every representation: the SPARQL query introduces two separate
variables that are both reached via `hasConnectionPoint`, the SHACL shape carries
`sh:qualifiedValueShape` entries (rather than a single class constraint), and the RDF
instance has two objects hanging off the same predicate.

Practical note on building the instance: each connection point below gets its *own*
`Connection` object, even though physically it's the same duct run on both sides. Try
reusing one `Connection` for both ends and `show_instance()` raises a `TypeError` -
`get_field_values(recursive=True)` hits its circular-reference guard on the shared
object partway through flattening it for template evaluation. The ontology has no
problem with a shared `Connection`; this is a limitation of the recursive
instance-to-dict step, not of the RDF model.

In [8]:
describe_class(entities.Fan)

### Fan
A piece of `Equipment` that causes a gas (e.g., air) to flow.

Fields: ['outlet_connection_point', 'inlet_connection_point']

--- SPARQL query (finds instances) ---
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX s223: <http://data.ashrae.org/standard223#>
SELECT DISTINCT * WHERE { ?inlet_connection_point rdf:type s223:InletConnectionPoint .
?name s223:hasConnectionPoint ?inlet_connection_point .
?outlet_connection_point rdf:type s223:OutletConnectionPoint .
?name s223:hasConnectionPoint ?outlet_connection_point .
?name rdf:type s223:Fan . }
--- SHACL shape (validates instances) ---
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix s223: <http://data.ashrae.org/standard223#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

s223:Fan a s223:Class,
        rdfs:Class,
        sh:NodeShape ;
    rdfs:label "Fan" ;
    rdfs:comment "A piece of `Equipment` that causes a gas (e.g., air) to flow." ;
    rdf

In [9]:
air = enumerationkinds.Air()
outlet = entities.OutletConnectionPoint(medium=air, connection=entities.Connection(medium=air))
inlet = entities.InletConnectionPoint(medium=air, connection=entities.Connection(medium=air))

fan = entities.Fan(outlet_connection_point=outlet, inlet_connection_point=inlet)
fan._name = "Fan_101"
show_instance(fan)

{'name': rdflib.term.URIRef('urn:tutorial#Fan_101'), 'outlet_connection_point': rdflib.term.URIRef('urn:tutorial#OutletConnectionPoint_1'), 'outlet_connection_point-_type': rdflib.term.Literal('OutletConnectionPoint'), 'outlet_connection_point-medium': rdflib.term.URIRef('urn:tutorial#Fluid-Air'), 'outlet_connection_point-medium-_type': rdflib.term.Literal('Air'), 'outlet_connection_point-connection': rdflib.term.URIRef('urn:tutorial#Connection_1'), 'outlet_connection_point-connection-_type': rdflib.term.Literal('Connection'), 'outlet_connection_point-connection-medium': rdflib.term.URIRef('urn:tutorial#Fluid-Air'), 'outlet_connection_point-connection-medium-_type': rdflib.term.Literal('Air'), 'outlet_connection_point-connection-medium-_circular_ref': rdflib.term.Literal('true', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#boolean')), 'inlet_connection_point': rdflib.term.URIRef('urn:tutorial#InletConnectionPoint_1'), 'inlet_connection_point-_type': rdflib.term.Literal

## 4. `Area` - a value, not an entity

`Area` is a `QuantifiableObservableProperty`: instead of only pointing at other typed
nodes, one of its fields (`value`) is a plain `float` that becomes an RDF `Literal`,
and the other two (`qk`, `unit`) are QUDT quantity-kind/unit references pinned as
fixed defaults rather than passed in at construction time. `Area(150.0)` only takes
the one value that's actually variable.

The RDF instance is typed `s223:QuantifiableObservableProperty`, not `s223:Area` -
`Area` is a hand-written subclass that fixes `qk` to `quantitykind:Area`, and its
`_semantic_type` points back at the real ontology base class so the generated triples
stay valid against the ontology (the same mechanism `s223-generated-classes-tutorial.ipynb`
introduces for `hvac_zone` in section 2).

In [10]:
describe_class(properties.Area)

### Area
This class is for instances of `QuantifiableProperty` for which numerical values are observed, like a temperature reading or a voltage measure.

Fields: ['qk', 'value', 'unit']

--- SPARQL query (finds instances) ---
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX quantitykind: <http://qudt.org/vocab/quantitykind/>
PREFIX s223: <http://data.ashrae.org/standard223#>
SELECT DISTINCT * WHERE { ?name s223:hasQuantityKind quantitykind:Area .
?name rdf:type s223:Area .
?name s223:hasValue ?value . }
--- SHACL shape (validates instances) ---
@prefix quantitykind: <http://qudt.org/vocab/quantitykind/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix s223: <http://data.ashrae.org/standard223#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

s223:Area a s223:Class,
        rdfs:Class,
        sh:NodeShape ;
    rdfs:label "Quantifiable observable property" ;
    rdfs:comment "This class is for instances 

In [11]:
area = properties.Area(150.0)
area._name = "Area_101"
show_instance(area)

{'name': rdflib.term.URIRef('urn:tutorial#Area_101'), 'qk': rdflib.term.URIRef('http://qudt.org/vocab/quantitykind/Area'), 'value': rdflib.term.Literal('150.0', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#double')), 'unit': rdflib.term.URIRef('http://qudt.org/vocab/unit/M2')}
--- RDF instance data ---
@prefix ns1: <http://data.ashrae.org/standard223#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<urn:tutorial#> a owl:Ontology .

<urn:tutorial#Area_101> a ns1:QuantifiableObservableProperty ;
    ns1:hasQuantityKind <http://qudt.org/vocab/quantitykind/Area> ;
    ns1:hasUnit <http://qudt.org/vocab/unit/M2> ;
    ns1:hasValue 1.5e+02 .




## Side by side

A rough measure of "how much structure" each object carries, across representations:

In [12]:
rows = []
for cls in [entities.Valve, entities.Duct, entities.Fan, properties.Area]:
    n_fields = len(cls.__dataclass_fields__)
    n_sparql_triples = cls.get_sparql_query(ontology='s223').count(" .\n")
    shacl = cls.generate_rdf_class_definition()
    n_property_shapes = shacl.count("sh:PropertyShape")
    rows.append((cls.__name__, n_fields, n_sparql_triples, n_property_shapes))

header = f"{'class':<10} {'fields':>8} {'sparql triples':>16} {'shacl property shapes':>24}"
print(header)
print("-" * len(header))
for name, f, t, p in rows:
    print(f"{name:<10} {f:>8} {t:>16} {p:>24}")

class        fields   sparql triples    shacl property shapes
-------------------------------------------------------------
Valve             0                0                        0
Duct              1                2                        0
Fan               2                4                        1
Area              3                2                        1


## Next steps

- `s223-generated-classes-tutorial.ipynb` - the same methods used here
  (`get_sparql_query`, `generate_rdf_class_definition`, `to_yaml`,
  `BMotifSession.evaluate`), introduced one at a time, plus the
  generated/hand-written override pattern `Area` relies on.
- `ontology-ingestion-tutorial.ipynb` - where these classes come from: parsing the
  ontology's own SHACL shapes into Python dataclasses.
- `docs/inference.md` - using SHACL-based type inference over raw RDF data, the
  reverse direction of the SHACL shapes shown above.